# 🔍 Notebook 5 — Explainable AI & Attention Visualization
### Project: Explainable Lightweight Edge-AI Framework for Early Cardiac Risk Prediction
**Week 3 | Task 1: Temporal Attention Maps, Grad-CAM & ECG Waveform Interpretability**

---
**Objectives:**
1. Extract attention weights from trained model
2. Visualize attention heatmaps on ECG waveforms
3. Implement Grad-CAM style ECG explainability
4. Generate waveform importance maps per risk class
5. Interpretable prediction summaries
---

## 📦 Cell 1 — Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.ndimage import zoom
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['axes.edgecolor']   = '#444'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#333'
plt.rcParams['grid.alpha']       = 0.5

os.makedirs('./outputs', exist_ok=True)

RISK_NAMES    = {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk', 3: 'Critical Risk'}
RISK_COLORS   = {0: '#00ff88', 1: '#ffd32a', 2: '#ff6b35', 3: '#ff4757'}
NUM_CLASSES   = 4
SAMPLING_RATE = 360
WINDOW_SIZE   = 180
BEFORE_R      = 90

print('✅ Libraries imported!')
print(f'   TensorFlow : {tf.__version__}')

---
## 📁 Cell 2 — Load Dataset & Models

In [ ]:
PROCESSED_PATH = r'D:\ECG-Cardiac-Risk-Prediction\notebooks\data\processed'

X = np.load(os.path.join(PROCESSED_PATH, 'X_segments.npy'))
y = np.load(os.path.join(PROCESSED_PATH, 'y_risk_labels.npy'))

X_dl  = X.reshape(X.shape[0], X.shape[1], 1).astype(np.float32)
y_cat = to_categorical(y, num_classes=NUM_CLASSES)

_, X_temp, _, y_temp = train_test_split(
    X_dl, y_cat, test_size=0.30, random_state=42, stratify=y
)
_, X_test, _, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42,
    stratify=np.argmax(y_temp, axis=1)
)
y_true = np.argmax(y_test, axis=1)

@tf.keras.utils.register_keras_serializable()
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight',
            shape=(input_shape[-1], 1), initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attention_bias',
            shape=(input_shape[1], 1), initializer='zeros', trainable=True)
        super(AttentionLayer, self).build(input_shape)
    def call(self, x):
        score   = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(x * weights, axis=1)
        return context, weights
    def compute_output_shape(self, input_shape):
        return [(input_shape[0], input_shape[-1]), (input_shape[0], input_shape[1], 1)]

custom_obj = {'AttentionLayer': AttentionLayer}
model           = tf.keras.models.load_model('./models/ecg_risk_model_final.keras',  custom_objects=custom_obj)
attention_model = tf.keras.models.load_model('./models/ecg_attention_model.keras',   custom_objects=custom_obj)

print('✅ Dataset & models loaded!')
print(f'   X_test shape : {X_test.shape}')
print(f'   y_test shape : {y_test.shape}')
print(f'   Model params : {model.count_params():,}')

---
## 🔍 Cell 3 — Extract Attention Weights

In [ ]:
print('🔄 Extracting attention weights...')
pred_probs, attn_weights = attention_model.predict(X_test, verbose=0)
y_pred       = np.argmax(pred_probs, axis=1)
attn_weights = attn_weights.squeeze(-1)   # (samples, timesteps)

# Upsample attention from BiLSTM timesteps back to original 180
attn_upsampled = np.array([
    zoom(w, WINDOW_SIZE / len(w)) for w in attn_weights
])

print(f'✅ Attention extracted!')
print(f'   Predictions shape    : {pred_probs.shape}')
print(f'   Raw attention shape  : {attn_weights.shape}')
print(f'   Upsampled attention  : {attn_upsampled.shape}  (matches ECG window)')
print(f'   Attention sum check  : {attn_weights[0].sum():.4f}  (should be ~1.0)')

---
## 📊 Cell 4 — Attention Heatmap on ECG (One per Risk Class)

In [ ]:
t_ms = np.arange(WINDOW_SIZE) / SAMPLING_RATE * 1000
fig  = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#0f0f0f')

for row, risk_id in enumerate(range(NUM_CLASSES)):
    candidates = np.where((y_true == risk_id) & (y_pred == risk_id))[0]
    if len(candidates) == 0:
        candidates = np.where(y_true == risk_id)[0]
    if len(candidates) == 0:
        continue
    idx  = candidates[0]
    ecg  = X_test[idx, :, 0]
    attn = attn_upsampled[idx]
    attn_norm = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    conf = pred_probs[idx, risk_id]
    color = RISK_COLORS[risk_id]

    # ECG + attention overlay
    ax1 = fig.add_subplot(NUM_CLASSES, 2, row * 2 + 1)
    for i in range(len(t_ms) - 1):
        ax1.fill_between([t_ms[i], t_ms[i+1]], [ecg[i], ecg[i+1]],
                         alpha=0.6, color=plt.cm.hot(attn_norm[i]))
    ax1.plot(t_ms, ecg, color='white', linewidth=1.0, alpha=0.9)
    ax1.axvline(x=BEFORE_R/SAMPLING_RATE*1000, color='cyan',
                linestyle='--', linewidth=1.2, label='R-peak')
    ax1.set_title(f'{RISK_NAMES[risk_id]} — ECG + Attention Heatmap  |  Conf: {conf:.3f}',
                  fontsize=10, color=color)
    ax1.set_ylabel('Amplitude', fontsize=9)
    ax1.legend(fontsize=8, facecolor='#1a1a2e', labelcolor='white')
    ax1.grid(True)

    # Attention weight plot
    ax2 = fig.add_subplot(NUM_CLASSES, 2, row * 2 + 2)
    ax2.plot(t_ms, attn_norm, color=color, linewidth=1.8)
    ax2.fill_between(t_ms, attn_norm, alpha=0.3, color=color)
    ax2.axvline(x=BEFORE_R/SAMPLING_RATE*1000, color='cyan',
                linestyle='--', linewidth=1.2)
    ax2.set_title(f'Temporal Attention Weights — {RISK_NAMES[risk_id]}',
                  fontsize=10, color=color)
    ax2.set_xlabel('Time (ms)', fontsize=9)
    ax2.set_ylabel('Attention', fontsize=9)
    ax2.grid(True)

fig.suptitle('Explainable AI — Attention Heatmaps on ECG Waveforms',
             fontsize=14, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig18_attention_heatmaps.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig18_attention_heatmaps.png')

---
## 📊 Cell 5 — Mean Attention per Risk Class

In [ ]:
t_ms = np.arange(WINDOW_SIZE) / SAMPLING_RATE * 1000
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for risk_id in range(NUM_CLASSES):
    ax    = axes[risk_id]
    color = RISK_COLORS[risk_id]
    idx   = np.where(y_true == risk_id)[0]
    if len(idx) == 0:
        ax.text(0.5, 0.5, 'No samples', ha='center', color='gray')
        continue
    mean_ecg  = X_test[idx, :, 0].mean(axis=0)
    std_ecg   = X_test[idx, :, 0].std(axis=0)
    mean_attn = attn_upsampled[idx].mean(axis=0)
    mean_attn = (mean_attn - mean_attn.min()) / (mean_attn.max() - mean_attn.min() + 1e-8)

    ax.plot(t_ms, mean_ecg, color=color, linewidth=2.0, label='Mean ECG')
    ax.fill_between(t_ms, mean_ecg-std_ecg, mean_ecg+std_ecg, alpha=0.2, color=color)
    for i in range(len(t_ms) - 1):
        ax.axvspan(t_ms[i], t_ms[i+1], alpha=mean_attn[i]*0.5, color='yellow', linewidth=0)
    ax.axvline(x=BEFORE_R/SAMPLING_RATE*1000, color='cyan', linestyle='--', linewidth=1.2, label='R-peak')
    ax.set_title(f'{RISK_NAMES[risk_id]}  (n={len(idx)})', fontsize=11, color=color)
    ax.set_xlabel('Time (ms)', fontsize=9)
    ax.set_ylabel('Amplitude', fontsize=9)
    ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=8)
    ax.grid(True)
    sm = ScalarMappable(cmap='YlOrRd', norm=Normalize(vmin=0, vmax=1))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Attention Weight', pad=0.01)

fig.suptitle('Mean ECG + Attention Shading per Risk Class\n(Yellow = High attention regions)',
             fontsize=13, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig19_mean_attention_per_class.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig19_mean_attention_per_class.png')

---
## 📊 Cell 6 — Grad-CAM Setup

In [ ]:
# Find last Conv1D layer
conv_layer_name = None
for layer in model.layers:
    if 'conv' in layer.name.lower():
        conv_layer_name = layer.name
print(f'   Using Conv layer: {conv_layer_name}')

grad_model = tf.keras.models.Model(
    inputs  = model.inputs,
    outputs = [model.get_layer(conv_layer_name).output, model.output]
)

def compute_gradcam(sample, class_idx):
    sample_tensor = tf.cast(sample[np.newaxis], tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(sample_tensor)
        conv_output, predictions = grad_model(sample_tensor)
        class_score = predictions[:, class_idx]
    grads    = tape.gradient(class_score, conv_output).numpy()[0]
    conv_out = conv_output.numpy()[0]
    weights  = np.mean(grads, axis=0)
    cam      = np.maximum(np.dot(conv_out, weights), 0)
    if cam.max() > 0:
        cam = cam / cam.max()
    return zoom(cam, WINDOW_SIZE / len(cam))

print('✅ Grad-CAM model ready!')

---
## 📊 Cell 7 — Grad-CAM Visualization

In [ ]:
t_ms = np.arange(WINDOW_SIZE) / SAMPLING_RATE * 1000
fig, axes = plt.subplots(NUM_CLASSES, 1, figsize=(18, 14), sharex=True)

for risk_id, ax in enumerate(axes):
    color      = RISK_COLORS[risk_id]
    candidates = np.where((y_true == risk_id) & (y_pred == risk_id))[0]
    if len(candidates) == 0:
        candidates = np.where(y_true == risk_id)[0]
    if len(candidates) == 0:
        ax.text(0.5, 0.5, 'No samples', ha='center', color='gray')
        continue
    idx  = candidates[0]
    ecg  = X_test[idx, :, 0]
    cam  = compute_gradcam(X_test[idx], risk_id)
    conf = pred_probs[idx, risk_id]

    ax.plot(t_ms, ecg, color='white', linewidth=1.2, zorder=3)
    for i in range(len(t_ms) - 1):
        ax.axvspan(t_ms[i], t_ms[i+1], alpha=cam[i]*0.6,
                   color=plt.cm.jet(cam[i]), linewidth=0)
    ax.axvline(x=BEFORE_R/SAMPLING_RATE*1000, color='cyan',
               linestyle='--', linewidth=1.2, label='R-peak')
    ax.set_title(f'{RISK_NAMES[risk_id]} — Grad-CAM  |  Confidence: {conf:.3f}',
                 fontsize=10, color=color)
    ax.set_ylabel('Amplitude', fontsize=9)
    ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)
    ax.grid(True)

axes[-1].set_xlabel('Time (ms)', fontsize=11)
fig.suptitle('Grad-CAM ECG Explainability — Class Activation Maps\n(Red=High importance | Blue=Low importance)',
             fontsize=13, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig20_gradcam_ecg.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig20_gradcam_ecg.png')

---
## 📊 Cell 8 — Attention Matrix Heatmap

In [ ]:
attn_matrix = np.zeros((NUM_CLASSES, WINDOW_SIZE))
for risk_id in range(NUM_CLASSES):
    idx = np.where(y_true == risk_id)[0]
    if len(idx) > 0:
        mean_attn = attn_upsampled[idx].mean(axis=0)
        mean_attn = (mean_attn - mean_attn.min()) / (mean_attn.max() - mean_attn.min() + 1e-8)
        attn_matrix[risk_id] = mean_attn

fig, ax = plt.subplots(figsize=(18, 5))
im = ax.imshow(attn_matrix, aspect='auto', cmap='hot',
               extent=[0, WINDOW_SIZE/SAMPLING_RATE*1000, -0.5, NUM_CLASSES-0.5])
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels([RISK_NAMES[i] for i in range(NUM_CLASSES)], fontsize=11)
ax.set_xlabel('Time (ms)', fontsize=11)
ax.axvline(x=BEFORE_R/SAMPLING_RATE*1000, color='cyan', linestyle='--',
           linewidth=2, label='R-peak')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
ax.set_title('Temporal Attention Heatmap Matrix — All Risk Classes\n(Brighter = Higher attention)',
             fontsize=13, color='white')
plt.colorbar(im, ax=ax, label='Normalized Attention')
plt.tight_layout()
plt.savefig('./outputs/fig21_attention_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig21_attention_matrix.png')

---
## 📊 Cell 9 — Clinical Prediction Report

In [ ]:
CLINICAL_HINTS = {
    0: 'Normal cardiac activity. No abnormalities detected.',
    1: 'Mild irregular activity. Possible supraventricular issue.',
    2: 'Ventricular abnormality detected. Review recommended.',
    3: 'CRITICAL: Life-threatening arrhythmia. Immediate attention!'
}
REGIONS = {'P-wave':(0,50), 'PR-seg':(50,90), 'QRS':(80,110), 'ST-seg':(110,140), 'T-wave':(130,180)}

for risk_id in range(NUM_CLASSES):
    candidates = np.where(y_true == risk_id)[0]
    if len(candidates) == 0:
        continue
    idx       = candidates[0]
    attn      = attn_upsampled[idx]
    attn_norm = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    pred_lbl  = y_pred[idx]
    conf      = pred_probs[idx, pred_lbl]

    region_attn = {r: attn_norm[s:e].mean() for r, (s,e) in REGIONS.items()}
    top_region  = max(region_attn, key=region_attn.get)

    print(f'\n╔══════════════════════════════════════════════════╗')
    print(f'║  ECG CARDIAC RISK REPORT — {RISK_NAMES[risk_id]:<21}║')
    print(f'╠══════════════════════════════════════════════════╣')
    print(f'║  Predicted : {RISK_NAMES[pred_lbl]:<36}║')
    print(f'║  Confidence: {conf*100:.1f}%{" "*35}║')
    print(f'║  Match     : {"✅ Correct" if risk_id==pred_lbl else "❌ Wrong":<36}║')
    print(f'╠══════════════════════════════════════════════════╣')
    print(f'║  Region Attention:                               ║')
    for r, sc in sorted(region_attn.items(), key=lambda x:-x[1]):
        bar = '█'*int(sc*15)
        print(f'║    {r:<10}: {sc:.3f} {bar:<15}              ║')
    print(f'║  Top region: {top_region:<36}║')
    print(f'╠══════════════════════════════════════════════════╣')
    print(f'║  Note: {CLINICAL_HINTS[pred_lbl]:<42}║')
    print(f'╚══════════════════════════════════════════════════╝')

---
## ✅ Cell 10 — Notebook 5 Summary

In [ ]:
print('=' * 58)
print('  🎉 Notebook 5 — Explainable AI Complete!')
print('=' * 58)
print('  Figures saved:')
print('   ✅ fig18_attention_heatmaps.png')
print('   ✅ fig19_mean_attention_per_class.png')
print('   ✅ fig20_gradcam_ecg.png')
print('   ✅ fig21_attention_matrix.png')
print()
print('  Methods used:')
print('   1. Temporal Attention Visualization')
print('   2. Mean Attention per Risk Class')
print('   3. Grad-CAM Style Explainability')
print('   4. Clinical Prediction Reports')
print()
print('  ➡️  Next: Notebook 6 — Cross-Dataset Validation')
print('=' * 58)